Step 1: Install Dependencies


In [ ]:
!pip install \
transformers==4.53.2 \
trl==0.19.1 \
peft==0.16.0 \
accelerate \
bitsandbytes \
datasets \
sentencepiece

Step 2: Hugging Face Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOKEN"))

Step 3: Import Libraries

In [ ]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import LoraConfig, PeftModel

from trl import SFTTrainer, SFTConfig

Step 4: Load Gemma-2-2B-IT

In [ ]:
model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Step 5: Load Dataset

Suppose your CSV file contains two columns:

*  context
*  question

In [ ]:

dataset = load_dataset(
    "csv",
    data_files="/content/questions.csv"
)

Step 6: Format Dataset

In [ ]:
def formatting_func(example):

    text = f"""
<start_of_turn>user
Generate one examination question from the following context.

Context:
{example['context']}
<end_of_turn>

<start_of_turn>model
{example['question']}
<end_of_turn>
"""

    return {"text": text}

dataset = dataset.map(formatting_func)

Step 7: Configure LoRA

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

Step 8: Configure training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./question_lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

Step 9: Create the trainer

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./question_lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step 10: Start training

In [ ]:
trainer.train()

Step,Training Loss
10,3.903000
20,1.747500
30,0.953200
40,0.718500
50,0.596700
60,0.562000
70,0.583700
80,0.503900
90,0.483000
100,0.449400


TrainOutput(global_step=375, training_loss=0.4641325880686442, metrics={'train_runtime': 232.988, 'train_samples_per_second': 12.863, 'train_steps_per_second': 1.61, 'total_flos': 2067707374636032.0, 'train_loss': 0.4641325880686442})

Step 11: Save the LoRA adapter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainer.save_model("/content/question_lora")

In [ ]:
trainer.save_model(
    "/content/drive/MyDrive/question_lora"
)

Step 12: Reload the base model

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
import transformers
import peft
import trl
import accelerate

print(transformers.__version__)
print(peft.__version__)
print(trl.__version__)
print(accelerate.__version__)

4.53.2
0.16.0
0.19.1
1.14.0


Step 13: Load your trained LoRA adapter

In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    "/content/question_lora"
)

Step 14: Merge the adapter into Gemma

In [ ]:
model = model.merge_and_unload()

model = model.cuda()

In [ ]:
print(model.device)
print(next(model.parameters()).device)

cuda:0
cuda:0


Step 15: Create the text generation pipeline

In [ ]:
from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

Device set to use cuda:0


Step 16: Test the model

In [ ]:
# context = """
# Mutual exclusion means that only one process can access a resource at a time.
# """

# context = """
# Blockchain is a distributed ledger technology that records transactions in immutable blocks linked together cryptographically.
# """

context = """
Newton's second law states that force is equal to mass multiplied by acceleration.
"""
prompt = f"""
<start_of_turn>user
Generate one examination question from the following context.

Context:
{context}
<end_of_turn>

<start_of_turn>model
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=False,
    temperature=None
)

generated = tokenizer.decode(
    outputs[0],
    skip_special_tokens=False
)

answer = generated.split("<start_of_turn>model")[-1].strip()

print(answer)

What is the relationship between force, mass, and acceleration?
<end_of_turn>
